# Gold: patient_360

Builds `azuredb_caresync.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`.

* **Driving table:** `silver.patients` (all patients kept, LEFT JOINs only)
* **Joins:** `silver.hospitals`, `silver.insurance_providers`, aggregated `silver.lab_results` by `patient_id`
* **Load type:** FULL rebuild — `CREATE OR REPLACE TABLE ... AS SELECT`

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredb_caresync.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE azuredb_caresync.gold.patient_360
COMMENT 'Gold: one row per patient with hospital, insurance and lab test summary'
AS
WITH lab_agg AS (
  -- Lab results aggregated to patient grain
  SELECT
    patient_id,
    COUNT(*)                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count,
    MAX(test_date)                         AS last_test_date
  FROM azuredb_caresync.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                    AS date_of_birth,  -- silver stores TIMESTAMP
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                                                         AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  ip.provider_name                                                                 AS insurance_provider_name,
  CAST(COALESCE(la.total_tests, 0) AS INT)                                         AS total_tests,
  CAST(COALESCE(la.abnormal_test_count, 0) AS INT)                                 AS abnormal_test_count,
  CAST(COALESCE(la.critical_test_count, 0) AS INT)                                 AS critical_test_count,
  la.last_test_date
FROM azuredb_caresync.silver.patients p
LEFT JOIN azuredb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN azuredb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id;

In [0]:
updated_df = spark.table("azuredb_caresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))